# Mini-projet — Gaz à effet de serre (NOAA GML)

**Cours :** Statistiques  
**Source des données :** [Global Monitoring Laboratory (NOAA)](https://gml.noaa.gov/ccgg/trends/data.html)

Ce notebook étudie l'évolution mensuelle de quatre gaz atmosphériques :
- `ch4_mm_gl.csv` — méthane (moyenne globale)
- `co2_mm_mlo.csv` — dioxyde de carbone (Mauna Loa, Hawaii)
- `n2o_mm_gl.csv` — protoxyde d'azote (moyenne globale)
- `sf6_mm_gl.csv` — hexafluorure de soufre (moyenne globale)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

# Affichage des graphiques dans le notebook
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

DATA_DIR = Path("data")

## Question 1 — Chargement et description des jeux de données

**Consigne :** charger les quatre jeux de données, préciser le nombre d'observations, la période étudiée, etc. Ne conserver que les variables `month` et `average`.

Les fichiers NOAA contiennent des lignes de commentaires (préfixées par `#`) qui précisent notamment les unités de mesure. Nous les lisons avec `comment="#"`.

In [ ]:
FILES = {
    "CH4 (méthane, global)": "ch4_mm_gl.csv",
    "CO2 (Mauna Loa)": "co2_mm_mlo.csv",
    "N2O (protoxyde d'azote, global)": "n2o_mm_gl.csv",
    "SF6 (hexafluorure de soufre, global)": "sf6_mm_gl.csv",
}

# Unités indiquées dans les fichiers NOAA
UNITS = {
    "CH4 (méthane, global)": "ppb (parties par milliard, nmol/mol)",
    "CO2 (Mauna Loa)": "ppm (parties par million, µmol/mol)",
    "N2O (protoxyde d'azote, global)": "ppb",
    "SF6 (hexafluorure de soufre, global)": "ppt (parties par trillion, fmol/mol)",
}


def load_noaa_csv(path: Path) -> pd.DataFrame:
    """Charge un fichier NOAA/GML (commentaires # + en-tête CSV)."""
    return pd.read_csv(path, comment="#", skipinitialspace=True)


def describe_dataset(name: str, path: Path) -> tuple[dict, pd.DataFrame]:
    """Retourne un résumé descriptif et le dataframe réduit à month + average."""
    raw = load_noaa_csv(path)

    # Normalisation du nom de colonne CO2 ("decimal date" -> "decimal")
    raw.columns = [c.strip().replace("decimal date", "decimal") for c in raw.columns]

    n_obs = len(raw)
    debut = f"{int(raw['year'].iloc[0])}-{int(raw['month'].iloc[0]):02d}"
    fin = f"{int(raw['year'].iloc[-1])}-{int(raw['month'].iloc[-1]):02d}"

    summary = {
        "Gaz": name,
        "Fichier": path.name,
        "Observations": n_obs,
        "Début": debut,
        "Fin": fin,
        "Unité": UNITS[name],
        "Min (average)": raw["average"].min(),
        "Max (average)": raw["average"].max(),
        "Moyenne (average)": raw["average"].mean().round(4),
    }

    # Conservation uniquement de month et average (consigne de l'énoncé)
    df = raw[["month", "average"]].copy()
    df.index = range(1, len(df) + 1)  # indice des observations (t = 1, 2, ...)
    df.index.name = "t"

    return summary, df


summaries = []
datasets = {}

for name, filename in FILES.items():
    summary, df = describe_dataset(name, DATA_DIR / filename)
    summaries.append(summary)
    datasets[name] = df

summary_df = pd.DataFrame(summaries).set_index("Gaz")
summary_df

In [ ]:
# Aperçu des jeux de données nettoyés (variables month et average uniquement)
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(f"Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    print(df.head(3))
    print("...")
    print(df.tail(3))

### Interprétation — Question 1

Les valeurs exactes sont calculées automatiquement par le code ci-dessus. À titre indicatif :

| Jeu de données | Observations | Période | Unité |
|---|---|---|---|
| **CH₄** (méthane global) | 511 | juillet 1983 → janvier 2026 | ppb |
| **CO₂** (Mauna Loa) | 818 | mars 1958 → avril 2026 | ppm |
| **N₂O** (global) | 301 | janvier 2001 → janvier 2026 | ppb |
| **SF₆** (global) | 343 | juillet 1997 → janvier 2026 | ppt |

**Remarques :**
- Les séries sont **mensuelles** : chaque ligne correspond à une moyenne centrée sur un mois.
- Le **CO₂** possède la série la plus longue (depuis 1958), référence historique de Keeling.
- Le **CH₄** couvre plus de 40 ans ; le **N₂O** et le **SF₆** ont des séries plus courtes.
- Toutes les séries montrent une **croissance globale** des concentrations (minimum nettement inférieur au maximum).
- Conformément à l'énoncé, seules les colonnes `month` (1–12) et `average` (concentration) sont conservées pour la suite de l'analyse. L'indice `t` numérote chronologiquement les observations de 1 à n.